## Taller de Programación en Python
### Profesor: Lucas Gómez Tobón

## Continuación de Pandas: Merge y Group By

En la sesión anterior aprendimos a crear, explorar, filtrar y transformar DataFrames individuales. En esta sesión daremos el siguiente paso: **combinar múltiples bases de datos** y **resumir información por grupos**, dos operaciones esenciales en cualquier flujo de análisis de datos.

### Objetivos de aprendizaje

Al finalizar esta sesión, usted será capaz de:

- Unir dos DataFrames usando `pd.merge()` con los 4 tipos de join (`left`, `right`, `inner`, `outer`).
- Validar que un pegue fue exitoso revisando llaves, duplicados y valores faltantes.
- Agrupar datos con `.groupby()` y aplicar funciones de agregación.
- Usar `.agg()` para aplicar múltiples funciones de agregación simultáneamente.
- Exportar los resultados de su análisis en formatos CSV, Excel y Pickle.

### Prerrequisitos

Este tutorial asume que usted ya completó la **Clase 9: Introducción a Pandas**, donde vimos:
- Creación de DataFrames, selección de filas/columnas con `.loc[]` e `.iloc[]`.
- Filtrado, ordenamiento, y estadísticas descriptivas.
- Concatenación de DataFrames con `pd.concat()`.

> **Recordatorio:** `pd.concat()` apila DataFrames vertical u horizontalmente cuando comparten los mismos índices o columnas. En esta clase veremos `pd.merge()`, que resuelve un problema diferente: unir tablas que comparten una **llave** (columna en común) pero no necesariamente las mismas filas.

### Unir bases de datos (`merge`)

#### ¿Por qué necesitamos `merge`?

En la sesión anterior aprendimos a concatenar filas y columnas con `pd.concat()`. Para hacer esto, es necesario que la cantidad de columnas o filas de los DataFrames a juntar sea la misma y que sus índices coincidan.

Sin embargo, en la práctica esto rara vez ocurre. Muchas veces querrá juntar bases de datos donde:
- No todas las llaves están presentes en ambas bases.
- A cada fila de una base le corresponden múltiples filas de la otra.
- Las columnas que identifican las filas tienen nombres diferentes en cada base.

Para estos casos utilizamos `pd.merge()`, que funciona de manera similar a un **JOIN en SQL**.

#### Sintaxis del `merge`

```python
pd.merge(left=left_dataframe, right=right_dataframe, on="columna_llave", how="left|right|inner|outer")
```

**Argumentos principales:**
- **`left`**: el DataFrame que va del lado izquierdo del pegue.
- **`right`**: el DataFrame que va del lado derecho del pegue.
- **`on`**: la columna (o lista de columnas) que sirve como llave para identificar qué filas de una tabla coinciden con qué filas de la otra. Esta llave debe identificar de forma única cada observación.
- **`how`**: el tipo de merge. Por defecto es `"inner"`. Las opciones son `"left"`, `"right"`, `"inner"` y `"outer"`, que exploraremos a continuación.

> **Nota sobre llaves con nombres diferentes:** Si la llave se llama distinto en cada DataFrame (por ejemplo, `"cc"` en uno y `"cedula"` en otro), puede usar los argumentos `left_on="cc"` y `right_on="cedula"` en lugar de `on`.

Veamos un ejemplo sencillo para entender cada tipo de merge:

In [1]:
import pandas as pd
import numpy as np

# ejemplos de pegues
left_dataframe = pd.DataFrame({"ID": [1,2,3,4], "left_side": "Izquierda"})
right_dataframe = pd.DataFrame({"ID": [3,4,5,6], "right_side": "Derecha"})

In [2]:
left_dataframe

,ID,left_side
0,1,Izquierda
1,2,Izquierda
2,3,Izquierda
3,4,Izquierda


In [3]:
right_dataframe

,ID,right_side
0,3,Derecha
1,4,Derecha
2,5,Derecha
3,6,Derecha


#### Left merge
En un Left merge lo que más nos interesa son los datos del lado IZQUIERDO a los cuales queremos pegarles columnas de una base de datos en el lado DERECHO.

Para hacer eso, cortamos las filas en el marco de datos DERECHO y pegamos partes en el marco de datos IZQUIERDO. Recuerde, nos preocupamos principalmente por el lado IZQUIERDO y solo queremos datos del lado DERECHO si tiene alguna de las mismas ID. Entonces, si algo en el marco de datos DERECHO no coincide o no existe, entonces tenemos que hacer cosas para mantener las columnas de la misma longitud. Lo hacemos agregando NaN para llenar el vacío o descartando algunas filas por completo.

En este ejemplo, el lado IZQUIERDO tiene los ID 1, 2, 3 y 4:
- El lado DERECHO no tiene ID 1 o 2, por lo que agregamos NaN porque necesitamos que las columnas tengan la misma longitud.
- El lado DERECHO tiene datos para los ID 3 y 4, así que lo agregamos como una nueva columna.
- El lado IZQUIERDO no tiene ID 5 o 6, por lo que no necesitamos esa información del DERECHO y se descarta.

<center>
<div>
<img src="img/left_merge.png" width="400"/>
</div>
</center>

In [4]:
# Left merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "left")

,ID,left_side,right_side
0,1,Izquierda,NaN
1,2,Izquierda,NaN
2,3,Izquierda,Derecha
3,4,Izquierda,Derecha


#### Right merge
Los Right merges funcionan igual que los Left merges, la diferencia es que nos preocupamos principalmente por el lado DERECHO y nos gustaría agregar datos desde el IZQUIERDO si tienen ID coincidentes.

<center>
<div>
<img src="img/right_merge.png" width="400"/>
</div>
</center>

In [5]:
# Right merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "right")

,ID,left_side,right_side
0,3,Izquierda,Derecha
1,4,Izquierda,Derecha
2,5,NaN,Derecha
3,6,NaN,Derecha


#### Inner merge
Con un Inner merge, cortamos ambos marcos de datos y solo pegamos las cosas que coinciden. Si una ID no está en ambos marcos de datos, no la mantenemos y no agregamos NaN.

<center>
<img src="img/inner_merge.png" width="400"/>
</center>

In [6]:
# Inner merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "inner")

,ID,left_side,right_side
0,3,Izquierda,Derecha
1,4,Izquierda,Derecha


#### Outer merge
Con un Outer merge, cortamos ambos marcos de datos y mantenemos todo de ambos lados. Luego agregamos NaN para llenar los espacios en blanco.

<center>
<img src="img/outer_merge.png" width="400"/>
</center>

In [7]:
# Outer merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "outer")

,ID,left_side,right_side
0,1,Izquierda,NaN
1,2,Izquierda,NaN
2,3,Izquierda,Derecha
3,4,Izquierda,Derecha
4,5,NaN,Derecha
5,6,NaN,Derecha


#### Ejemplo práctico: uniendo bases de datos de restaurantes

Ahora aplicaremos `merge` con datos reales. Importaremos dos bases de datos sobre usuarios que califican restaurantes en internet:
- **`payment`**: informa el método de pago favorito de cada cliente.
- **`profile`**: describe características sociodemográficas de los clientes.

Nuestro objetivo es juntar ambas bases para tener un perfil completo de cada cliente.

> **Nota sobre sintaxis:** Hasta ahora hemos usado `pd.merge(left=..., right=...)`. También es posible llamar `.merge()` directamente desde un DataFrame, por ejemplo: `profile.merge(payment, on="userID", how="left")`. Ambas formas son equivalentes; en este ejemplo usaremos ambas para que las conozca.

In [8]:
# Importe bases de datos
payment = pd.read_csv("Datos/userpayment.csv")
profile = pd.read_csv("Datos/userprofile.csv")

In [9]:
# Analicemos la estructura de las bases
payment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   userID    177 non-null    object
 1   Upayment  177 non-null    object
dtypes: object(2)
memory usage: 2.9+ KB


In [10]:
payment.head()

,userID,Upayment
0,U1001,cash
1,U1002,cash
2,U1003,cash
3,U1004,cash
4,U1004,bank_debit_cards


In [11]:
profile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   userID            138 non-null    object 
 1   latitude          138 non-null    float64
 2   longitude         138 non-null    float64
 3   smoker            138 non-null    object 
 4   drink_level       138 non-null    object 
 5   dress_preference  138 non-null    object 
 6   ambience          138 non-null    object 
 7   transport         138 non-null    object 
 8   marital_status    138 non-null    object 
 9   hijos             138 non-null    object 
 10  birth_year        138 non-null    int64  
 11  interest          138 non-null    object 
 12  personality       138 non-null    object 
 13  religion          138 non-null    object 
 14  activity          138 non-null    object 
 15  color             138 non-null    object 
 16  weight            138 non-null    int64  
 1

In [12]:
profile.head()

,userID,latitude,longitude,smoker,drink_level,dress_preference,ambience,transport,marital_status,hijos,birth_year,interest,personality,religion,activity,color,weight,budget,height
0,U1001,22.139997,-100.978803,false,abstemious,informal,family,on foot,single,independent,1989,variety,thrifty-protector,none,student,black,69,medium,1.77
1,U1002,22.150087,-100.983325,false,abstemious,informal,family,public,single,independent,1990,technology,hunter-ostentatious,Catholic,student,red,40,low,1.87
2,U1003,22.119847,-100.946527,false,social drinker,formal,family,public,single,independent,1989,none,hard-worker,Catholic,student,blue,60,low,1.69
3,U1004,18.867000,-99.183000,false,abstemious,informal,family,public,single,independent,1940,variety,hard-worker,none,professional,green,44,medium,1.53
4,U1005,22.183477,-100.959891,false,abstemious,no preference,family,public,single,independent,1992,none,thrifty-protector,Catholic,student,black,65,medium,1.69


Note que ambas bases tienen diferente número de observaciones, `profile` tiene 133 observaciones mientras que `payment` tiene 177 clientes. Esto quiere decir que, aunque profile es una caracterización más completa de los clientes, payment tiene más observaciones. Adicionalmente, ninguna de las bases tiene NAs.

Al parecer `userID` corresponde a la llave/identificador de cada cliente/fila. Revisemos que no hayan duplicados!

In [13]:
payment.userID.duplicated().sum()

44

In [14]:
profile.userID.duplicated().sum()

0

Mientras que `profile` no tiene duplicados, `payment` tiene 44 duplicados, vamos a revisarlos. Al parecer ambas bases tienen los mismos clientes, lo que pasa es que algunos tienen más de un tipo de método de pago.

In [15]:
# Devolver todos los duplicados
payment.loc[payment.userID.duplicated(False),]

,userID,Upayment
3,U1004,cash
4,U1004,bank_debit_cards
12,U1012,cash
13,U1012,bank_debit_cards
14,U1013,MasterCard-Eurocard
...,...,...
155,U1117,cash
159,U1121,cash
160,U1121,bank_debit_cards
170,U1133,bank_debit_cards


**Decisión analítica:** Vamos a eliminar los duplicados dejando solo la primera observación por usuario. Esto implica asumir que el primer método de pago registrado es el preferido.

Esta es una decisión que usted como analista debe tomar conscientemente. Otras alternativas serían:
- Quedarse con el último registro (`keep="last"`).
- Concatenar los métodos de pago en una sola cadena de texto.
- No eliminar duplicados y aceptar que el merge generará más filas.

Lo importante es **documentar y justificar** la decisión que tome.

In [16]:
# Eliminamos duplicados dejando solo la primera observación por usuario
payment = payment.drop_duplicates(subset=["userID"], keep="first").reset_index(drop=True)

Como la base que más nos interesa es la de `profile` vamos a hacer que esta sea nuestra base de la IZQUIERDA y hacer un LEFT merge

In [17]:
df = pd.merge(left = profile, right = payment, on = "userID", how = "left")
df.head()

,userID,latitude,longitude,smoker,drink_level,dress_preference,ambience,transport,marital_status,hijos,birth_year,interest,personality,religion,activity,color,weight,budget,height,Upayment
0,U1001,22.139997,-100.978803,false,abstemious,informal,family,on foot,single,independent,1989,variety,thrifty-protector,none,student,black,69,medium,1.77,cash
1,U1002,22.150087,-100.983325,false,abstemious,informal,family,public,single,independent,1990,technology,hunter-ostentatious,Catholic,student,red,40,low,1.87,cash
2,U1003,22.119847,-100.946527,false,social drinker,formal,family,public,single,independent,1989,none,hard-worker,Catholic,student,blue,60,low,1.69,cash
3,U1004,18.867000,-99.183000,false,abstemious,informal,family,public,single,independent,1940,variety,hard-worker,none,professional,green,44,medium,1.53,cash
4,U1005,22.183477,-100.959891,false,abstemious,no preference,family,public,single,independent,1992,none,thrifty-protector,Catholic,student,black,65,medium,1.69,cash


Debemos revisar que todos los elementos en profile hayan encontrado un match exacto en payment

In [18]:
df["Upayment"].isna().sum()

5

Upa! Tenemos 5 NAs. Eso quiere decir que hay 5 usuarios/clientes en profile que no están en payment! Revisemos

In [19]:
usuarios_faltantes = df.loc[df["Upayment"].isna(), "userID"].values
usuarios_faltantes

array(['U1024', 'U1025', 'U1088', 'U1122', 'U1130'], dtype=object)

In [20]:
payment["userID"].isin(usuarios_faltantes).sum()

0

In [21]:
# En efecto, estos 5 usuarios no están en la base de payment
set(profile["userID"]) - set(payment["userID"])

{'U1024', 'U1025', 'U1088', 'U1122', 'U1130'}

In [22]:
# Sin embargo, en la base de profile sí están todos los usuarios de payment
set(payment["userID"]) - set(profile["userID"])

set()

In [23]:
profile.shape

(138, 19)

In [24]:
payment.shape

(133, 2)

> Receta para hacer un pegue exitoso:

1. **Diseñar el pegue.** Antes de correr el código se debe definir:
    - Se desean añadir observaciones o variables? Para pegar observaciones (filas) se puede utilizar la función `pd.concat()` y para pegar variables (columnas) la función `pd.merge()`.
    - Identificar el data frame que recibe la información y el data frame que entrega la información. En los pegues siempre hay un data frame que es la estructura base sobre el cual se añadirá nueva información. Identificar el rol de cada base antes del pegue es clave para evitar generar duplicados o perdida de información.
    - Identificar cuál es el nivel de observación de cada base del pegue. Usualmente el nivel de información puede ser entendido a través de los identificadores únicos (o llaves) de cada base. El nivel de información describe la población que es descrita en cada fila, por ejemplo: individuos, familias, ciudades, departamentos, países, etc. 
    - Identificar las llaves de cada base.
2. **Validar las llaves.** Para cada base del pegue validar que:
    - La llave es un identificador único de las observaciones (no hay duplicados).
    - La llave no tiene valores faltantes (`NA`).
    - Cada llave de la base de la izquierda mapea unicamente una llave de la base de la derecha y viceversa. 
    - Los tipos de las variables de las llaves son iguales entre bases. Es decir, si la llave `x` de la base de la izquierda es `str`, la llave `x` de la base de la derecha debe ser también `str`.
    - Antes de hacer el pegue revisar que todas las observaciones de la base de la derecha estén en la base de la izquierda. En caso de que no, llevar el conteo y corroborar este número en el paso 3.
3. **Validar que el pegue fue exitoso.** Una vez se hizo el pegue se debe validar que:
    - El número de observaciones de la base original no se incrementó (en el caso de añadir columnas).
    - No se generaron NAs en las variables añadidas. En caso de haber NAs tienen una explicación clara: la observación faltante proviene de ausencia de la variable en la base de la derecha (última instrucción del paso 2).

---

### Agrupación y agregación con `.groupby()`

Ahora que sabemos combinar bases de datos, el siguiente paso natural es **resumir y analizar** la información combinada. En análisis de datos, es muy común querer calcular estadísticas **por categoría**: el promedio de ventas por región, el total de pacientes por hospital, la calificación media por tipo de restaurante, etc.

Para esto, pandas ofrece el método `.groupby()`, que permite dividir los datos en grupos según una o más columnas y luego aplicar una función de agregación a cada grupo.

In [25]:
df = pd.read_excel("Datos/ejemplo_groupby.xlsx")
df

,animal,age,weight,length
0,hamster,1,7,8
1,alligator,9,13,6
2,hamster,4,8,9
3,cat,13,12,1
4,snake,14,11,8
5,cat,10,8,9
6,hamster,2,10,5
7,cat,4,14,6
8,cat,14,9,6
9,snake,7,11,6


Note que tenemos un `dataframe` con cuatro tipos de animales: 
- alligators (cocodrilos 🐊)
- cats (gatos 🐱)
- snakes (serpientes 🐍)
- hamsters (hamsters 🐹)

Cada una de las filas indican un chequeo en el veterinario donde se registra edad, peso y largo del animal. Por ende, usted como investigador quiere estudiar algunas estadísticas descriptivas por especie. Por ejemplo ¿Cuál es el peso promedio de cada especie?

In [26]:
# El primer paso es agrupar por animal
animal_groups = df.groupby("animal")

In [27]:
animal_groups

> **¿Qué es este objeto?** `.groupby()` no devuelve un DataFrame directamente, sino un objeto especial llamado `DataFrameGroupBy`. Este objeto "sabe" cómo están agrupados los datos, pero para obtener un resultado visible debemos aplicarle una **función de agregación** como `.mean()`, `.sum()`, `.count()`, etc.

In [28]:
# Veamos la conformación de cada uno de los grupos. ¿En qué filas aparece cada animal?
animal_groups.groups

{'alligator': [1, 13], 'cat': [3, 5, 7, 8, 12], 'hamster': [0, 2, 6, 10, 11], 'snake': [4, 9]}

In [29]:
# El segundo paso es aplicar una funcion agregadora
# ¿Cuál es la media del peso por especie?
animal_groups["weight"].mean()

animal
alligator    13.5
cat          10.4
hamster       9.0
snake        11.0
Name: weight, dtype: float64

Visualmente, lo que sucedió fue lo siguiente:

1. Se agrupa los valores únicos de la columna animal.
<center>
<img src = "img/groupby1.jpg" width = "400">
</center>

2. La segmentación de cada grupo se vería de la siguiente manera
<center>
<img src = "img/groupby2.jpg" width = "400">
</center>

3. Se le asignan las otras variables/columnas a cada grupo
<center>
<img src = "img/groupby3.jpg" width = "400">
</center>

4. Se aplica la función agregadora `.mean()` sobre la columna `weight` de cada grupo.
<center>
<img src = "img/groupby4.jpg" width = "400">
</center>


#### Funciones de agregación más comunes

| Función | Descripción | Ejemplo |
|---------|-------------|---------|
| `.mean()` | Promedio | Peso promedio por especie |
| `.median()` | Mediana | Edad mediana por especie |
| `.std()` | Desviación estándar | Variabilidad del peso |
| `.min()` | Valor mínimo | Largo mínimo por especie |
| `.max()` | Valor máximo | Largo máximo por especie |
| `.sum()` | Suma total | Total de peso por especie |
| `.count()` | Conteo de valores no nulos | Cantidad de registros por especie |

Probemos algunas de estas funciones:

In [30]:
# Probemos otros ejemplos
# ¿Cuál es la edad mediana por animal?
df.groupby("animal")["age"].median()

animal
alligator     8.0
cat          10.0
hamster       2.0
snake        10.5
Name: age, dtype: float64

In [31]:
# ¿Cuál es el largo máximo por animal?
df.groupby("animal")["length"].max()

animal
alligator    6
cat          9
hamster      9
snake        8
Name: length, dtype: int64

In [32]:
# ¿Cuál es la desviación estándar del peso por animal?
df.groupby("animal")["weight"].std()

animal
alligator    0.707107
cat          2.509980
hamster      1.414214
snake        0.000000
Name: weight, dtype: float64

#### Ejercicio integrador: Merge + GroupBy

Ahora combinaremos las dos herramientas que hemos aprendido en un solo análisis. Vamos a:
1. **Unir** (`merge`) una base de datos de calificaciones de restaurantes con otra que contiene información sobre el tipo de parqueadero de cada restaurante.
2. **Agrupar** (`.groupby()`) las calificaciones por tipo de parqueadero para responder la pregunta: **¿influye el tipo de parqueadero en la percepción que tienen los clientes sobre un restaurante?**

Los tipos de parqueadero son: `none`, `public`, `valet parking` y `yes` (tiene parqueadero propio).

In [33]:
# 1. Importe los datos
ratings = pd.read_csv("Datos/rating_final.csv")
parking = pd.read_csv("Datos/chefmozparking.csv")

In [34]:
# Inspeccione los datos
ratings.head()

,userID,placeID,rating,food_rating,service_rating
0,U1077,135085,2,2,2
1,U1077,135038,2,2,1
2,U1077,132825,2,2,2
3,U1077,135060,1,2,2
4,U1068,135104,1,1,2


In [35]:
# Se puede ver que userID se refiere al identificador de usuario que calificó al restaurante placeID. 
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1161 entries, 0 to 1160
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userID          1161 non-null   object
 1   placeID         1161 non-null   int64 
 2   rating          1161 non-null   int64 
 3   food_rating     1161 non-null   int64 
 4   service_rating  1161 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 45.5+ KB


In [36]:
# Inspeccionemos la base de parking
parking.head()

,placeID,parking_lot
0,135111,public
1,135110,none
2,135109,none
3,135108,none
4,135107,none


In [37]:
# Para cada restaurante (placeID) se tiene una descripción del tipo de parqueadero.
# Estudiemos cuántos tipos de parqueaderos tiene cada restaurante
parking.placeID.value_counts().describe()

count    675.00000
mean       1.04000
std        0.20353
min        1.00000
25%        1.00000
50%        1.00000
75%        1.00000
max        3.00000
Name: count, dtype: float64

In [38]:
# Veamos la proporción de tipos de parqueaderos
parking.parking_lot.value_counts(normalize = True)

parking_lot
none                 0.495726
yes                  0.247863
public               0.145299
street               0.045584
fee                  0.031339
valet parking        0.029915
validated parking    0.004274
Name: proportion, dtype: float64

In [39]:
# En general cada restaurante tiene un sólo tipo de parqueadero pero hay algunos que tienen más de 1 tipo
# Pregunta: ¿Cuál es la variable con la que queremos hacer el pegue?
# ¿Qué tipo de pegue queremos hacer?

In [40]:
# Queremos hacer el pegue con la variable placeID.
# Debemos verificar que ambas variables estén en el mismo formato
ratings.placeID.dtype

dtype('int64')

In [41]:
parking.placeID.dtype

dtype('int64')

In [42]:
# Esto debe ser True siempre
ratings.placeID.dtype == parking.placeID.dtype

True

In [43]:
# Vamos a hacer un left join porque queremos tener absolutamente todas las calificaciones de los restaurantes
ratings = ratings.merge(parking, on = "placeID", how = "left")

In [44]:
# Veamos que tan bueno estuvo el pegue

# ¿Cuál es la cantidad de NAs o valores faltantes por variable?
ratings.isna().sum()

userID            0
placeID           0
rating            0
food_rating       0
service_rating    0
parking_lot       0
dtype: int64

In [45]:
# ¿Cuál es la proporción de NAs o valores faltantes por variable?
ratings.isna().mean()

userID            0.0
placeID           0.0
rating            0.0
food_rating       0.0
service_rating    0.0
parking_lot       0.0
dtype: float64

In [46]:
# Note que hay 0 NAs en parking_lot, sin embargo hay algunos parqueaderos con el valor "none" (texto).
# Esto es importante: NaN (Not a Number) y el string "none" NO son lo mismo.
# - NaN es un valor faltante que pandas reconoce automáticamente.
# - "none" es simplemente un texto que indica que el restaurante no tiene parqueadero.
# pandas no los trata igual:
print("¿NaN es igual a None?", np.nan == None)  # False
print()
print("Valores únicos de parking_lot:")
print(ratings["parking_lot"].value_counts())

¿NaN es igual a None? False

Valores únicos de parking_lot:
parking_lot
none             561
yes              389
public           182
valet parking     29
Name: count, dtype: int64


**¿Como hacemos para analizar las variables de rating a la luz del tipo de parqueo?**

In [47]:
ratings.groupby("parking_lot")[["rating", "food_rating", "service_rating"]].mean() \
    .round(2).sort_values("service_rating", ascending = False)

,rating,food_rating,service_rating
parking_lot,,,
valet parking,1.34,1.34,1.34
none,1.20,1.21,1.10
yes,1.21,1.21,1.09
public,1.15,1.22,1.02


¿Qué pasaría si no quisiera tener solo la media sino otras estadísticas más completas?

#### Método .agg()
El método .agg() se puede utilizar después de aplicar un método .groupby() en pandas para realizar operaciones de agregación en los datos de cada grupo.

La sintaxis general de la función .groupby() es la siguiente:
```python
dataframe.groupby(columnas).agg(funciones)
```
Donde:
- dataframe: el DataFrame al que se aplicará la función `groupby()`.
- columnas: la(s) columna(s) que se utilizarán para agrupar los datos.
- funciones: la(s) operación(es) de agregación que se aplicarán a los datos agrupados.

Por ejemplo, para calcular la media, el máximo y el mínimo de las columnas de rating del DataFrame agrupado por la columna 'parking_lot', se puede utilizar la siguiente sintaxis:

In [48]:
ratings.groupby("parking_lot")[["rating", "food_rating", "service_rating"]].agg(["min", "mean", "max"])

rating               food_rating               service_rating  \
                 min      mean max         min      mean max            min   
parking_lot                                                                   
none               0  1.203209   2           0  1.212121   2              0   
public             0  1.148352   2           0  1.219780   2              0   
valet parking      0  1.344828   2           0  1.344828   2              0   
yes                0  1.208226   2           0  1.208226   2              0   

                             
                   mean max  
parking_lot                  
none           1.098039   2  
public         1.021978   2  
valet parking  1.344828   2  
yes            1.092545   2

También es posible utilizar varias columnas para agrupar los datos y aplicar diferentes operaciones de agregación a diferentes columnas. Por ejemplo:

In [49]:
ratings.groupby("parking_lot").agg({'rating': ['mean', 'max'], 'food_rating': 'std', 
                                     "service_rating": lambda x: np.percentile(x, 50)})

rating     food_rating service_rating
                   mean max         std       <lambda>
parking_lot                                           
none           1.203209   2    0.783488            1.0
public         1.148352   2    0.804713            1.0
valet parking  1.344828   2    0.813979            2.0
yes            1.208226   2    0.799700            1.0

In [50]:
# Otra sintaxis, en vez de un diccionario, usar tuplas
ratings.groupby("parking_lot").agg(rating_media = ("rating", 'mean'), 
                                   rating_maximo = ("rating", 'max'),
                                   service_rating_mediana = ("service_rating", lambda x: np.percentile(x, 50)))

,rating_media,rating_maximo,service_rating_mediana
parking_lot,,,
none,1.203209,2,1.0
public,1.148352,2,1.0
valet parking,1.344828,2,2.0
yes,1.208226,2,1.0


In [51]:
pd.set_option('display.max_columns', None)
ratings.groupby("parking_lot")[["rating", "food_rating", "service_rating"]].describe()

rating                                              food_rating  \
               count      mean       std  min  25%  50%  75%  max       count   
parking_lot                                                                     
none           561.0  1.203209  0.777857  0.0  1.0  1.0  2.0  2.0       561.0   
public         182.0  1.148352  0.768850  0.0  1.0  1.0  2.0  2.0       182.0   
valet parking   29.0  1.344828  0.768852  0.0  1.0  2.0  2.0  2.0        29.0   
yes            389.0  1.208226  0.770148  0.0  1.0  1.0  2.0  2.0       389.0   

                                                           service_rating  \
                   mean       std  min  25%  50%  75%  max          count   
parking_lot                                                                 
none           1.212121  0.783488  0.0  1.0  1.0  2.0  2.0          561.0   
public         1.219780  0.804713  0.0  1.0  1.0  2.0  2.0          182.0   
valet parking  1.344828  0.813979  0.0  1.0  2.0  2.0  2.0           29.0   
yes            1.208226  0.799700  0.0  1.0  1.0  2.0  2.0          389.0   

                                                            
                   mean       std  min  25%  50%  75%  max  
parking_lot                                                 
none           1.098039  0.799115  0.0  0.0  1.0  2.0  2.0  
public         1.021978  0.764951  0.0  0.0  1.0  2.0  2.0  
valet parking  1.344828  0.813979  0.0  1.0  2.0  2.0  2.0  
yes            1.092545  0.787578  0.0  0.0  1.0  2.0  2.0

---

### Exportar los resultados del análisis

Ya tenemos nuestro análisis completo: unimos bases de datos, agrupamos por categoría y calculamos estadísticas. El último paso en cualquier flujo de trabajo es **guardar los resultados** para compartirlos o usarlos posteriormente.

Pandas ofrece varias opciones para exportar datos a diferentes formatos de archivo. Veamos las más comunes:

#### 1. Exportar a CSV con `.to_csv()`

In [52]:
# Creamos un dataframe para el ejemplo
df = pd.DataFrame({'col1': [1, 2], 'col2': [3, 4]})
df

,col1,col2
0,1,3
1,2,4


In [53]:
df.to_csv('Datos/df_coma.csv', sep = ',')

In [54]:
df.to_csv('Datos/df_punto_coma.csv', sep = ';')

#### 2. Exportar a Excel con [.to_excel()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_excel.html)

In [55]:
df.to_excel('Datos/df.xlsx', sheet_name = 'Prueba', index = False)

#### 3. Exportar a Pickle con [.to_pickle()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_pickle.html)

Pickle es un formato binario de Python que permite guardar objetos de forma compacta. Es especialmente útil para conjuntos de datos grandes porque:
- **Ocupa menos espacio** que CSV o Excel.
- **Preserva los tipos de datos** (no pierde información sobre fechas, categorías, etc.).
- **Es más rápido** de leer y escribir para archivos grandes.

**Desventajas:**
- No es legible por humanos (no se puede abrir en un editor de texto).
- Solo funciona con Python (no se puede abrir en Excel directamente).
- **Advertencia de seguridad:** solo deserialice archivos pickle de fuentes confiables, ya que pueden ejecutar código arbitrario.

Veamos un ejemplo comparando los tamaños de archivo:

In [56]:
df.to_pickle('Datos/df.gzip', compression = 'gzip')

In [57]:
pd.read_pickle('Datos/df.gzip', compression = "gzip")

,col1,col2
0,1,3
1,2,4


In [58]:
import os
for i in os.listdir("Datos/"):
    if "df" in i:
        print("El archivo", i, "pesa", os.stat("Datos/" + i).st_size, "bytes")

El archivo df.xlsx pesa 4942 bytes
El archivo df.gzip pesa 479 bytes
El archivo df_coma.csv pesa 23 bytes
El archivo df_punto_coma.csv pesa 23 bytes


In [59]:
import numpy as np

accidentes = pd.read_csv('Datos/info_accidentes.csv')
accidentes.to_pickle('Datos/info_accidentes.gzip', compression = 'gzip')

for i in os.listdir("Datos/"):
    if "info_accidentes" in i:
        print("El archivo", i, "pesa", np.round(os.stat("Datos/" + i).st_size/ (1024 * 1024), 0), "megabytes")

El archivo info_accidentes.gzip pesa 0.0 megabytes
El archivo info_accidentes.csv pesa 5.0 megabytes


---

### Resumen de la clase

En esta sesión aprendimos tres herramientas fundamentales para el análisis de datos:

| Herramienta | ¿Para qué sirve? |
|-------------|-------------------|
| `pd.merge()` | Unir dos DataFrames por una o más columnas en común (llaves), con 4 tipos de join: `left`, `right`, `inner`, `outer`. |
| `.groupby()` | Dividir los datos en grupos según una columna categórica y aplicar funciones de agregación (`.mean()`, `.sum()`, `.std()`, etc.). |
| `.agg()` | Aplicar múltiples funciones de agregación a múltiples columnas en un solo paso. |
| `.to_csv()`, `.to_excel()`, `.to_pickle()` | Exportar los resultados del análisis en diferentes formatos. |

**Recuerde siempre:**
- Antes de un merge: valide las llaves (sin duplicados, sin NAs, mismo tipo de dato).
- Después de un merge: verifique que el número de filas sea el esperado y revise los NAs generados.
- Un buen análisis no solo ejecuta código, sino que **verifica cada paso** y **documenta las decisiones** tomadas.